# Module 13 — SQL for Data Science (Phase 1: Topics 6–7)

SQL is on your CV and it's the **#1 most-tested skill** in data interviews after
Python. Data lives in databases; you must fetch and shape it *before* pandas ever
sees it. We use **SQLite** (built into Python — nothing to install) and load our
customer data so every query runs live.

Goals:
- The **logical order** SQL executes in (not the order you write it).
- `WHERE` vs `HAVING`, aggregation, `GROUP BY`.
- All **JOIN** types with predictable row counts.
- **Subqueries**, **CTEs**, and **window functions** (the senior-level stuff).
- How SQL and pandas mirror each other.

In [1]:
import sqlite3
import pandas as pd

# Load our CSVs into an in-memory SQLite database
con = sqlite3.connect(":memory:")
customers = pd.read_csv("../data/customers_clean.csv")
customers.to_sql("customers", con, index=False, if_exists="replace")

# A tiny lookup table so we can practice JOINs
cities = pd.DataFrame({
    "city": ["Nairobi", "Mombasa", "Kisumu", "Nakuru", "Eldoret"],
    "region": ["Central", "Coast", "Nyanza", "Rift Valley", "Rift Valley"],
    "population_k": [4397, 1208, 610, 570, 475],
})
cities.to_sql("cities", con, index=False, if_exists="replace")

def q(sql):
    "Run SQL and return a DataFrame (our little query helper)."
    return pd.read_sql_query(sql, con)

print("tables:", q("SELECT name FROM sqlite_master WHERE type='table'")["name"].tolist())
q("SELECT customer_id, age, city, plan, monthly_spend, churn FROM customers LIMIT 5")

tables: ['customers', 'cities']


,customer_id,age,city,plan,monthly_spend,churn
0,1,38.0,Kisumu,Basic,186.65,0
1,2,25.0,Nakuru,Basic,76.83,0
2,3,35.0,Nairobi,Basic,37.36,1
3,4,44.0,Mombasa,Premium,23.34,0
4,5,18.0,Nairobi,Basic,12.17,1


## 13.1 The clause order you WRITE vs the order SQL RUNS

You write: `SELECT … FROM … WHERE … GROUP BY … HAVING … ORDER BY … LIMIT`.
SQL logically **executes** in this order:

1. **FROM** (+ JOIN) — get and combine the tables
2. **WHERE** — filter individual **rows**
3. **GROUP BY** — bucket rows into groups
4. **HAVING** — filter **groups**
5. **SELECT** — choose/compute columns
6. **ORDER BY** — sort
7. **LIMIT** — cut

Knowing this explains *why* you can't use a `SELECT` alias in `WHERE` (WHERE runs
first) and why `HAVING` is for aggregated conditions.

In [2]:
# WHERE filters rows BEFORE grouping
q('''
SELECT city, plan, monthly_spend
FROM customers
WHERE monthly_spend > 100 AND plan = 'Premium'
ORDER BY monthly_spend DESC
LIMIT 5
''')

,city,plan,monthly_spend
0,Nairobi,Premium,236.80
1,Nairobi,Premium,202.72
2,Mombasa,Premium,202.05
3,Kisumu,Premium,186.77
4,Nairobi,Premium,141.83


## 13.2 Aggregation: GROUP BY + WHERE vs HAVING

`WHERE` filters **rows** (before grouping). `HAVING` filters **groups** (after
aggregation). This distinction is a classic interview question.

In [3]:
# Churn rate & average spend per plan (AVG of a 0/1 column = a rate — same trick as pandas)
q('''
SELECT plan,
       COUNT(*)                AS n_customers,
       ROUND(AVG(monthly_spend), 2) AS avg_spend,
       ROUND(AVG(churn), 3)    AS churn_rate
FROM customers
GROUP BY plan
ORDER BY churn_rate DESC
''')

,plan,n_customers,avg_spend,churn_rate
0,Basic,249,68.28,0.406
1,Premium,95,63.58,0.242
2,Standard,156,72.65,0.212


In [4]:
# HAVING: keep only cities with more than 50 customers
q('''
SELECT city, COUNT(*) AS n
FROM customers
GROUP BY city
HAVING COUNT(*) > 50
ORDER BY n DESC
''')

,city,n
0,Nairobi,208
1,Mombasa,102
2,Nakuru,70
3,Kisumu,69
4,Eldoret,51


## 13.3 JOINs — combining tables

Join `customers` to `cities` to bring in `region`. Ask two questions every time:
**on what key?** and **which join type?**
- `INNER JOIN` — only rows with a match in both.
- `LEFT JOIN` — all left rows; unmatched right = NULL.

In [5]:
q('''
SELECT c.customer_id, c.city, ci.region, c.plan, c.monthly_spend
FROM customers AS c
LEFT JOIN cities AS ci
       ON c.city = ci.city
LIMIT 5
''')

,customer_id,city,region,plan,monthly_spend
0,1,Kisumu,Nyanza,Basic,186.65
1,2,Nakuru,Rift Valley,Basic,76.83
2,3,Nairobi,Central,Basic,37.36
3,4,Mombasa,Coast,Premium,23.34
4,5,Nairobi,Central,Basic,12.17


In [6]:
# Aggregate AFTER joining: churn rate by region
q('''
SELECT ci.region,
       COUNT(*)             AS n_customers,
       ROUND(AVG(c.churn),3) AS churn_rate
FROM customers AS c
LEFT JOIN cities AS ci ON c.city = ci.city
GROUP BY ci.region
ORDER BY churn_rate DESC
''')

,region,n_customers,churn_rate
0,Nyanza,69,0.362
1,Central,208,0.341
2,Rift Valley,121,0.306
3,Coast,102,0.235


## 13.4 Subqueries — a query inside a query

Use a subquery to compare rows against an aggregate — e.g. customers who spend
**above the overall average**.

In [7]:
q('''
SELECT customer_id, plan, monthly_spend
FROM customers
WHERE monthly_spend > (SELECT AVG(monthly_spend) FROM customers)
ORDER BY monthly_spend DESC
LIMIT 5
''')

,customer_id,plan,monthly_spend
0,347,Standard,427.55
1,54,Standard,244.80
2,7,Basic,242.23
3,254,Basic,237.17
4,151,Premium,236.80


## 13.5 CTEs (`WITH`) — readable, multi-step queries

A **Common Table Expression** names an intermediate result, so complex logic reads
top-to-bottom instead of nesting subqueries. Interviewers love clean CTEs.

In [8]:
q('''
WITH plan_stats AS (
    SELECT plan,
           AVG(monthly_spend) AS plan_avg_spend
    FROM customers
    GROUP BY plan
)
SELECT c.customer_id, c.plan, c.monthly_spend,
       ROUND(p.plan_avg_spend, 2) AS plan_avg,
       ROUND(c.monthly_spend - p.plan_avg_spend, 2) AS vs_plan_avg
FROM customers AS c
JOIN plan_stats AS p ON c.plan = p.plan
ORDER BY vs_plan_avg DESC
LIMIT 5
''')

,customer_id,plan,monthly_spend,plan_avg,vs_plan_avg
0,347,Standard,427.55,72.65,354.90
1,7,Basic,242.23,68.28,173.95
2,151,Premium,236.80,63.58,173.22
3,54,Standard,244.80,72.65,172.15
4,254,Basic,237.17,68.28,168.89


## 13.6 Window functions — aggregate WITHOUT collapsing rows

The senior-level tool. A window function computes across a set of rows **related to
the current row** while keeping every row. Syntax:
`FUNC(...) OVER (PARTITION BY ... ORDER BY ...)`.

- `ROW_NUMBER / RANK / DENSE_RANK` — ranking within groups.
- `SUM/AVG ... OVER (PARTITION BY ...)` — group stat attached to each row.
- `LAG / LEAD` — previous/next row (great for time series & churn diffs).

In [9]:
# Rank customers by spend WITHIN each plan (top spender per plan = rank 1)
q('''
SELECT customer_id, plan, monthly_spend,
       RANK() OVER (PARTITION BY plan ORDER BY monthly_spend DESC) AS spend_rank
FROM customers
ORDER BY plan, spend_rank
LIMIT 9
''')

,customer_id,plan,monthly_spend,spend_rank
0,7,Basic,242.23,1
1,254,Basic,237.17,2
2,406,Basic,235.45,3
3,132,Basic,225.26,4
4,90,Basic,215.49,5
5,271,Basic,194.69,6
6,20,Basic,188.73,7
7,1,Basic,186.65,8
8,25,Basic,183.64,9


In [10]:
# Each customer's spend vs their plan's average, on the SAME row (no GROUP BY collapse)
q('''
SELECT customer_id, plan, monthly_spend,
       ROUND(AVG(monthly_spend) OVER (PARTITION BY plan), 2) AS plan_avg,
       ROUND(monthly_spend - AVG(monthly_spend) OVER (PARTITION BY plan), 2) AS diff
FROM customers
ORDER BY diff DESC
LIMIT 5
''')

,customer_id,plan,monthly_spend,plan_avg,diff
0,347,Standard,427.55,72.65,354.90
1,7,Basic,242.23,68.28,173.95
2,151,Premium,236.80,63.58,173.22
3,54,Standard,244.80,72.65,172.15
4,254,Basic,237.17,68.28,168.89


## 13.7 SQL ↔ pandas (the same ideas, two languages)

| Task | SQL | pandas |
|---|---|---|
| filter rows | `WHERE x > 5` | `df[df.x > 5]` |
| choose columns | `SELECT a, b` | `df[["a","b"]]` |
| group + aggregate | `GROUP BY g` + `AVG()` | `df.groupby("g").mean()` |
| join | `JOIN ... ON` | `df.merge(other, on=...)` |
| sort | `ORDER BY x DESC` | `df.sort_values("x", ascending=False)` |
| top-n | `LIMIT 5` | `df.head(5)` / `nlargest` |
| distinct | `SELECT DISTINCT` | `df.drop_duplicates()` |

Being fluent in **both**, and knowing they mirror each other, is exactly what data
roles want.

## 13.8 Mini-exercises (write the SQL, then check with `q(...)`)

1. Average `income` per `region` (join required), highest first.
2. Customers whose `tenure_months` is above their **own plan's** average tenure
   (window function or CTE).
3. For each city, the **rank** of customers by `monthly_spend`; return only rank 1.
4. Count churned vs stayed per plan as **two columns** (hint: `SUM(CASE WHEN ...)`).

In [11]:
# Example solution to #4 — conditional aggregation (pivot in SQL)
q('''
SELECT plan,
       SUM(CASE WHEN churn = 1 THEN 1 ELSE 0 END) AS churned,
       SUM(CASE WHEN churn = 0 THEN 1 ELSE 0 END) AS stayed
FROM customers
GROUP BY plan
''')

,plan,churned,stayed
0,Basic,101,148
1,Premium,23,72
2,Standard,33,123


In [12]:
con.close()   # always close the connection when done
print("connection closed — module complete")

connection closed — module complete


## Summary

- SQL **executes** `FROM→WHERE→GROUP BY→HAVING→SELECT→ORDER BY→LIMIT`.
- **WHERE** filters rows; **HAVING** filters groups.
- Master all **JOINs** and predict row counts.
- **Subqueries** and **CTEs** structure complex logic; CTEs read cleaner.
- **Window functions** (`RANK`, `SUM() OVER`, `LAG/LEAD`) aggregate without
  collapsing rows — the skill that marks a strong candidate.
- SQL and pandas are two dialects of the same ideas.

Next: **Module 14 — APIs & Web Scraping** (getting data that isn't handed to you).